# Hands-On Lab: Production Data Preprocessing & Leakage Prevention
Accompanies **Chapter 5: Data Processing Pipelines in Practice**.

In this lab, we build a leak-free data preprocessing pipeline on the peer-to-peer lending dataset (`loan_data_train.csv`).
We will:
1. Identify storage types vs true information types using the **Difference Test**.
2. Clean dirty strings and formatted percentages.
3. Handle missing values using domain medians.
4. Encode categorical columns using the 3-step dummy pattern (explicit missing, clubbing rare categories, dropping baseline to avoid dummy trap).
5. Assemble the pipeline using `utils.ml_utils` without data leakage.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add utils to sys.path
sys.path.append(os.path.abspath('../../utils'))
from ml_utils import (
    VarNumericCleaner,
    VarImputer,
    VarCategoricalEncoder,
    DataFramePipeline
)

# Load raw loan dataset
data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

### 1. Data Type Audit
Let's inspect `df.info()` to see columns where storage types disagree with information types.
Notice that `Interest.Rate` is stored as an `object` because of trailing `%` characters, and `Debt.To.Income.Ratio` also contains `%`.

In [ ]:
print(df.dtypes)
print("\nSample of dirty percentage columns:")
print(df[['Interest.Rate', 'Debt.To.Income.Ratio']].head())

### 2. Feature Cleaning & Coercion
We use `VarNumericCleaner` to strip percentage signs and commas, converting them to native `float64`.

In [ ]:
cleaner = VarNumericCleaner(columns=['Interest.Rate', 'Debt.To.Income.Ratio'], strip_chars=['%', '$', ','])
df_cleaned = cleaner.fit_transform(df)

print(df_cleaned[['Interest.Rate', 'Debt.To.Income.Ratio']].dtypes)
df_cleaned[['Interest.Rate', 'Debt.To.Income.Ratio']].head()

### 3. The 3-Step Categorical Encoding Pattern
For categorical columns (e.g., `Home.Ownership`, `Employment.Length`):
1. Replace missing values with `__missing__`.
2. Group rare levels (< 1% frequency) into `__low_freq_cat__`.
3. Create $k-1$ dummies to avoid the Dummy Variable Trap.

In [ ]:
cat_encoder = VarCategoricalEncoder(
    columns=['Home.Ownership'],
    missing_token='__missing__',
    rare_threshold=0.01,
    drop_first=True
)
df_encoded = cat_encoder.fit_transform(df_cleaned)
dummy_cols = [c for c in df_encoded.columns if c.startswith('Home.Ownership_')]
print("Generated Dummy Columns:")
print(dummy_cols)
df_encoded[dummy_cols].head()

### 4. Operational Leakage Audit
Remember our operational audit from Chapter 5:
`Amount.Funded.By.Investors` is populated *after* investors view the application. At application submission time, it is non-existent.
We must drop it before modeling.

In [ ]:
if 'Amount.Funded.By.Investors' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['Amount.Funded.By.Investors'])
    print("Successfully dropped 'Amount.Funded.By.Investors' to prevent post-decision leakage.")